# 2.3 Multi-Agent — Orchestrating Specialised Agents

## What you will learn in this notebook

- What a **multi-agent system** is and why you'd use one
- How to create **sub-agents** with focused, specialised capabilities
- How to **wrap sub-agents as tools** so a main agent can call them
- How the **orchestrator pattern** routes tasks to the right specialist
- The tradeoffs between single-agent and multi-agent architectures

---

### Why multi-agent?

A single agent with many tools works well for simple tasks. But as complexity grows, problems emerge:

- **Context overload**: giving one agent 20 tools fills the context window with tool descriptions, leaving less room for reasoning
- **Role confusion**: a model given conflicting tools (medical advice + financial advice + legal advice) may mix domains incorrectly
- **No specialisation**: one agent can't be simultaneously expert in everything

**Multi-agent systems** solve this by splitting responsibilities:

```
Single agent (monolith):                Multi-agent (modular):

┌─────────────────────────┐             ┌──────────────────┐
│  One Agent              │             │  Main Agent       │ ← orchestrator
│  tools: [20 tools]      │             │  tools: [         │
│  can't specialise       │             │    call_agent_1,  │
│  high context pressure  │             │    call_agent_2   │
└─────────────────────────┘             │  ]                │
                                        └────────┬──────────┘
                                                 │
                              ┌──────────────────┼──────────────────┐
                              ▼                                     ▼
                    ┌──────────────────┐               ┌──────────────────┐
                    │  Sub-Agent 1     │               │  Sub-Agent 2     │
                    │  tools: [√ only] │               │  tools: [² only] │
                    │  expert in √     │               │  expert in ²     │
                    └──────────────────┘               └──────────────────┘
```

### The orchestrator pattern

The **main agent** (orchestrator) does NOT have domain tools directly. Instead, it has tools that **call sub-agents**. Its job is:
1. Understand the user's request
2. Decide which sub-agent to delegate to
3. Pass the right input to that sub-agent
4. Synthesise the result for the user

The sub-agents are specialists — they don't know about each other.

In [ ]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

---
## Section 1 — Creating Sub-Agents

In [ ]:
# ============================================================
# CELL 2: Define Specialised Tools
# ============================================================
# Each sub-agent gets exactly ONE tool — it is a pure specialist.
# This keeps each sub-agent's context small and its role unambiguous.
#
# square_root → only knows how to take square roots
# square      → only knows how to square numbers
#
# In a real system these might be:
#   @tool def search_flights(...)  → for a flights specialist agent
#   @tool def book_hotel(...)      → for a hotels specialist agent
#   @tool def fetch_weather(...)   → for a weather specialist agent

from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [ ]:
# ============================================================
# CELL 3: Create the Sub-Agents
# ============================================================
# Each sub-agent is a full agent — it has its own LLM, its own tool,
# and its own reasoning loop. It just has a very narrow domain.
#
# No system_prompt needed here — the tool descriptions are clear
# enough for the model to know what to do when asked.
#
# No checkpointer needed — sub-agents are stateless workers.
# They receive one question, answer it, and return. The main agent
# handles the overall conversation memory if needed.

from langchain.agents import create_agent

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]   # Only knows how to take square roots
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]        # Only knows how to square numbers
)

---
## Section 2 — Calling Sub-Agents as Tools

The bridge between the main agent and sub-agents is **wrapping each sub-agent in a `@tool`**. This means:
- The main agent sees `call_subagent_1` and `call_subagent_2` as ordinary tools
- It calls them like any other tool — passing a number, getting a result
- Under the hood, each `call_subagent_*` function runs a full agent invocation

The main agent never knows it's talking to another agent — it just sees tools.

In [ ]:
# ============================================================
# CELL 4: Wrap Sub-Agents as Callable Tools
# ============================================================
# Each wrapper tool:
#   1. Takes a float x from the main agent
#   2. Constructs a natural language question
#   3. Calls the sub-agent with that question
#   4. Extracts the final answer from the response
#   5. Returns it to the main agent as a string
#
# Key design decisions:
#
#   f"Calculate the square root of {x}"
#   → We construct a full natural language prompt for the sub-agent.
#     This is important — sub-agents are full agents with LLMs,
#     not just functions. They expect natural language input.
#
#   response["messages"][-1].content
#   → Same pattern as always — the last message is the final answer.
#
#   return type is float but we return a string
#   → The actual return is a text string from the LLM ("√456 ≈ 21.35").
#     The main agent reads this string and presents it to the user.
#     The type hint is aspirational — in practice the LLM may add
#     prose around the number. For pure numeric output, consider
#     structured output on the sub-agent.
#
# The docstrings are critical:
#   "Call subagent 1 in order to calculate the square root..."
#   The main agent reads these to decide which wrapper to call.

from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content  # Extract sub-agent's final answer

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content

# ============================================================
# Create the Main Agent (Orchestrator)
# ============================================================
# The main agent has NO maths tools directly — only the two
# sub-agent wrappers. Its job is purely to route and coordinate.
#
# system_prompt makes its role explicit: "call subagents".
# This prevents the main agent from trying to answer maths
# questions itself (which LLMs do poorly).

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],   # Sub-agent wrappers, not raw tools
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number."
)

---
## Section 3 — Testing the Multi-Agent System

In [ ]:
# ============================================================
# CELL 5: Ask the Main Agent a Question
# ============================================================
# Trace of what happens under the hood:
#
#   main_agent receives: "What is the square root of 456?"
#
#   [main_agent LLM]
#     → decides to call call_subagent_1(x=456)
#
#   [call_subagent_1 runs]
#     → subagent_1.invoke({"messages": ["Calculate the square root of 456"]})
#
#     [subagent_1 LLM]
#       → decides to call square_root(x=456)
#       → square_root returns 21.354...
#       → subagent_1 replies: "The square root of 456 is approximately 21.35"
#
#     call_subagent_1 returns: "The square root of 456 is approximately 21.35"
#
#   [main_agent LLM] reads the result
#     → main_agent replies: "The square root of 456 is approximately 21.35"
#
# There are TWO full agent loops here — main + sub — but you only
# interact with the main_agent interface.

question = "What is the square root of 456?"

response = main_agent.invoke(
    {"messages": [HumanMessage(content=question)]}
)

In [ ]:
# ============================================================
# CELL 6: Inspect the Main Agent's Message Trace
# ============================================================
# The main agent's trace shows 4 messages (tool call pattern):
#   [0] HumanMessage     → "What is the square root of 456?"
#   [1] AIMessage        → tool call: call_subagent_1(x=456)
#   [2] ToolMessage      → "The square root of 456 is approximately 21.35"
#                           (this is what the sub-agent returned)
#   [3] AIMessage        → main agent's final reply
#
# The sub-agent's internal messages are NOT visible here —
# they happen inside call_subagent_1 and are discarded after.
# Only the final output of the sub-agent appears as a ToolMessage.

from pprint import pprint

pprint(response)

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Sub-agent** | A full agent with a narrow, specialised toolset — expert in one domain |
| **Wrapper tool** | A `@tool` that calls a sub-agent — bridges orchestrator to specialist |
| **Orchestrator** | Main agent with no domain tools — only sub-agent wrapper tools |
| **Docstrings matter** | Main agent reads wrapper docstrings to decide which sub-agent to call |
| **Two LLM calls** | Each sub-agent call triggers its own full LLM reasoning loop |
| **Hidden internals** | Sub-agent's internal messages don't appear in main agent's trace |

### Tradeoffs

| | Single Agent | Multi-Agent |
|---|---|---|
| **Latency** | Lower (one LLM call) | Higher (multiple LLM calls) |
| **Cost** | Lower | Higher |
| **Complexity** | Lower | Higher |
| **Specialisation** | Limited | High |
| **Context efficiency** | Poor (all tools visible) | Good (each agent has few tools) |
| **Best for** | Simple, focused tasks | Complex, multi-domain tasks |

### Next steps
- **2.4 Wedding Planners** — a realistic multi-agent system combining MCP, memory, and state
- **Bonus: RAG** — giving agents access to private document collections via vector search